In [1]:
import torch
import sqlite3
import os
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
from bitsandbytes.optim import PagedAdamW32bit

In [ ]:
MODEL_PATH     = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
SPIDER_DB_DIR  = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/database"
SPIDER_TABLES  = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"
OUTPUT_DIR     = "./qwen-spider-finetuned"
USE_BF16       = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16       = torch.cuda.is_available() and not USE_BF16
USE_CPU        = not torch.cuda.is_available()

LR              = 1e-5
NUM_EPOCHS      = 3
GRAD_ACCUM      = 8
WARMUP_STEPS    = 20
MAX_PROMPT_LEN  = 605
MAX_NEW_TOKENS  = 71
NUM_SAMPLES     = 4       
LOGGING_STEPS   = 50
EVAL_STEPS      = 200
SAVE_STEPS      = 200
SAVE_TOTAL_LIMIT = 3

In [3]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

def format_schema_with_fk(db):
    col_names = db["column_names_original"]
    lines = []
    for i, table in enumerate(db["table_names_original"]):
        cols = [col[1] for col in col_names if col[0] == i]
        lines.append(f"  {table}({', '.join(cols)})")
    if db.get("foreign_keys"):
        lines.append("  Foreign keys:")
        for fk in db["foreign_keys"]:
            c1 = col_names[fk[0]]
            c2 = col_names[fk[1]]
            t1 = db["table_names_original"][c1[0]]
            t2 = db["table_names_original"][c2[0]]
            lines.append(f"    {t1}.{c1[1]} → {t2}.{c2[1]}")
    return "\n".join(lines)

schema_index = {db["db_id"]: format_schema_with_fk(db) for db in tables_data}

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [5]:
dataset    = load_dataset("spider")
train_data = dataset["train"]
val_data   = dataset["validation"]

FEW_SHOT = """
Example 1:
Database: concert_singer
Schema:
  singer(Singer_ID, Name, Country, Age)
Question: How many singers are from USA?
SQL: SELECT COUNT(*) FROM singer WHERE Country = 'USA';

Example 2:
Database: concert_singer  
Schema:
  concert(concert_ID, Name, Stadium_ID)
  stadium(Stadium_ID, Name, Capacity)
Question: What are the names of all stadiums?
SQL: SELECT Name FROM stadium;
"""

def build_prompt(example):
    schema = schema_index.get(example["db_id"], "")
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert SQL assistant. "
                "Output ONLY the SQL query with no explanation, "
                "no markdown, no backticks, no comments. "
                "Just the raw SQL query ending with a semicolon.\n\n"
                f"{FEW_SHOT}"
            )
        },
        {
            "role": "user",
            "content": (
                f"Database: {example['db_id']}\n"
                f"Schema:\n{schema}\n\n"
                f"Question: {example['question']}"
            )
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Testar uso máximo de VRAM
# train_data = sorted(train_data, key=lambda ex: len(tokenizer(build_prompt(ex))["input_ids"]), reverse=True)

In [6]:
def extract_sql(text):
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    match = re.search(r"(SELECT|INSERT|UPDATE|DELETE|WITH).+", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).split("\n\n")[0].strip()
    return text.strip()

def execution_reward(db_id, pred_sql, gold_sql):
    db_path = os.path.join(SPIDER_DB_DIR, db_id, f"{db_id}.sqlite")
    try:
        conn     = sqlite3.connect(db_path)
        cur      = conn.cursor()
        cur.execute(pred_sql)
        pred_res = set(cur.fetchall())
        cur.execute(gold_sql)
        gold_res = set(cur.fetchall())
        conn.close()
        return 1.0 if pred_res == gold_res else 0.0
    except Exception:
        return 0.0

In [ ]:
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False

base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head"        
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

optimizer = PagedAdamW32bit(model.parameters(), lr=LR)

total_steps = (len(train_data) * NUM_EPOCHS) // GRAD_ACCUM
scheduler   = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [8]:
def evaluate(model, val_data, max_examples=200):
    model.eval()
    correct = 0
    total   = min(max_examples, len(val_data))
    device  = next(model.parameters()).device
    for example in list(val_data)[:total]:
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_tokens = output[0][inputs["input_ids"].shape[1]:]
        pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
        correct   += execution_reward(example["db_id"], pred_sql, example["query"])
    model.train()
    return correct / total

In [9]:
device         = next(model.parameters()).device
global_step    = 0
best_eval      = 0.0
saved_checkpoints = []

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    total_loss   = 0.0
    total_reward = 0.0

    for i, example in enumerate(tqdm(train_data, desc=f"Epoch {epoch+1}", disable=False)):
        prompt = build_prompt(example)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(device)

        rewards        = []
        log_probs_list = []
        torch.cuda.empty_cache()

        for _ in range(NUM_SAMPLES):
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.8,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            gen_tokens = output[0][inputs["input_ids"].shape[1]:]
            pred_sql   = extract_sql(tokenizer.decode(gen_tokens, skip_special_tokens=True))
            reward     = execution_reward(example["db_id"], pred_sql, example["query"])
            rewards.append(reward)

            labels = output.clone()
            labels[0, :inputs["input_ids"].shape[1]] = -100
            out = model(input_ids=output, labels=labels)
            log_probs_list.append(-out.loss)

        # REINFORCE com baseline
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
        baseline       = rewards_tensor.mean()
        advantages     = rewards_tensor - baseline

        policy_loss = torch.tensor(0.0, requires_grad=True).to(device)
        for adv, lp in zip(advantages, log_probs_list):
            policy_loss = policy_loss + (-adv.to(device) * lp)
        policy_loss = policy_loss / NUM_SAMPLES
        total_loss_step = policy_loss

        (total_loss_step / GRAD_ACCUM).backward()

        total_loss   += total_loss_step.detach().item()
        total_reward += rewards_tensor.mean().detach().item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()  
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOGGING_STEPS == 0:
                print(f"Step {global_step} | loss: {total_loss/global_step:.4f} | reward: {total_reward/global_step:.4f} | lr: {scheduler.get_last_lr()[0]:.2e}")

            if global_step % EVAL_STEPS == 0:
                ex = evaluate(model, val_data)
                print(f"Step {global_step} | Eval EX: {ex:.4f}")

            if global_step % SAVE_STEPS == 0:
                save_path = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                model.save_pretrained(save_path)
                saved_checkpoints.append(save_path)
                if len(saved_checkpoints) > SAVE_TOTAL_LIMIT:
                    import shutil
                    shutil.rmtree(saved_checkpoints.pop(0))

model.save_pretrained(f"{OUTPUT_DIR}/final")
print("Treino concluído.")

Epoch 1:   0%|          | 0/7000 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


KeyboardInterrupt: 